In [50]:
import cv2
import mediapipe as mp
import numpy as np
import time
import tkinter as tk
from pynput.mouse import Controller, Button

mouse = Controller()

In [51]:
# Get screen resolution without pyautogui
root = tk.Tk()
screen_w, screen_h = root.winfo_screenwidth(), root.winfo_screenheight()
root.destroy()

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)
mp_draw = mp.solutions.drawing_utils


I0000 00:00:1790520764.172592  138367 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1790520764.174578  146808 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.2.2), renderer: AMD Radeon Graphics (radeonsi, renoir, ACO, DRM 3.64, 7.2.5-200.fc44.x86_64)


W0000 00:00:1790520764.212312  146796 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1790520764.248003  146797 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [52]:
def get_landmark_list(hand_landmarks, frame_w, frame_h):
    lm_list = []
    for id, lm in enumerate(hand_landmarks.landmark):
        cx, cy = int(lm.x * frame_w), int(lm.y * frame_h)
        lm_list.append((id, cx, cy))
    return lm_list

In [53]:
def fingers_up(lm_list):
    """
    Returns list of 5 values [thumb, index, middle, ring, pinky]
    1 = finger up, 0 = finger down
    """
    fingers = []
    tips_ids = [4, 8, 12, 16, 20]

    # Thumb (compare x-coordinates since thumb moves sideways)
    if lm_list[tips_ids[0]][1] > lm_list[tips_ids[0] - 1][1]:
        fingers.append(1)
    else:
        fingers.append(0)

    # Other 4 fingers (compare y-coordinates)
    for id in range(1, 5):
        if lm_list[tips_ids[id]][2] < lm_list[tips_ids[id] - 2][2]:
            fingers.append(1)
        else:
            fingers.append(0)

    return fingers

In [54]:
smoothening = 7
prev_x, prev_y = 0, 0
curr_x, curr_y = 0, 0

INVERT_X = False  # toggle this if left/right is still backwards
frame_reduction = 100  # margin so hand doesn't need to reach screen edges
click_delay = 0.3
last_click_time = 0

In [ ]:
cap = cv2.VideoCapture(0)
cap.set(3, 640)
cap.set(4, 480)

cv2.namedWindow("Virtual Mouse", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Virtual Mouse", 1280, 720)

global prev_x, prev_y, curr_x, curr_y, last_click_time

while True:
    success, img = cap.read()
    if not success:
        break

    img = cv2.flip(img, 1)
    frame_h, frame_w, _ = img.shape
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)

    if INVERT_X:
        final_x = screen_w - curr_x
    else:
        final_x = curr_x

    mouse.position = (final_x, curr_y)
    
    if results.multi_hand_landmarks:
        hand_landmarks = results.multi_hand_landmarks[0]
        mp_draw.draw_landmarks(img, hand_landmarks, mp_hands.HAND_CONNECTIONS)

        lm_list = get_landmark_list(hand_landmarks, frame_w, frame_h)

        if len(lm_list) != 0:
            x1, y1 = lm_list[8][1], lm_list[8][2]   # index finger tip
            x2, y2 = lm_list[12][1], lm_list[12][2] # middle finger tip

            fingers = fingers_up(lm_list)

            # Draw active tracking region
            cv2.rectangle(img, (frame_reduction, frame_reduction),
                          (frame_w - frame_reduction, frame_h - frame_reduction),
                          (255, 0, 255), 2)

            # ---- MODE 1: Only Index finger up -> Move cursor ----
            if fingers[1] == 1 and fingers[2] == 0:
                x3 = np.interp(x1, (frame_reduction, frame_w - frame_reduction), (0, screen_w))
                y3 = np.interp(y1, (frame_reduction, frame_h - frame_reduction), (0, screen_h))

                curr_x = prev_x + (x3 - prev_x) / smoothening
                curr_y = prev_y + (y3 - prev_y) / smoothening

                mouse.position = (screen_w - curr_x, curr_y)  
                prev_x, prev_y = curr_x, curr_y

            # ---- MODE 2: Index + Middle finger up -> Click ----
            if fingers[1] == 1 and fingers[2] == 1:
                distance = np.hypot(x2 - x1, y2 - y1)
                cv2.line(img, (x1, y1), (x2, y2), (0, 255, 0), 3)

                if distance < 40:  # fingers close together = click
                    current_time = time.time()
                    if current_time - last_click_time > click_delay:
                        mouse.click(Button.left, 1)
                        cv2.circle(img, ((x1+x2)//2, (y1+y2)//2), 15, (0, 255, 0), cv2.FILLED)
                        last_click_time = current_time

    cv2.imshow("Virtual Mouse", img)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [7]:
cap.release()
cv2.destroyAllWindows()
hands.close()